# Deploying Agents in Production

Companion notebook for the [Deploying Agents lesson](https://ml-viz-ruby.vercel.app/courses/agent-design-patterns/11-deploying-agents).

**The idea in one sentence.** A production agent needs **guardrails the prompt can't
override** — hard **step and cost ceilings**, **checkpointing** to resume after a crash,
and awareness that **cost and latency scale with trajectory length**.

The three production essentials, from scratch:

- **Bounded loops:** enforce `max_steps` and `max_cost` in the *runtime*, not the
  prompt — an agent that "wants" 50 steps stops at your ceiling.
- **Cost & latency scale with steps:** every step is tokens and seconds; parallelism
  helps latency but not cost.
- **Checkpointing:** persist state each step so a crash resumes instead of restarting.

We **validate that the runtime ceilings bind regardless of the agent's intent**, then
cover the gotchas.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Dark style matching the site theme.
plt.style.use('dark_background')
plt.rcParams.update({
    'axes.edgecolor': '#475569',
    'axes.labelcolor': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'axes.titlecolor': '#e2e8f0',
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'grid.color': '#2e3347',
    'savefig.facecolor': '#0f1117',
})
BRAND = '#6366f1'
TEAL = '#14b8a6'
ROSE = '#f43f5e'
YELLOW = '#eab308'

rng = np.random.default_rng(0)

## 1. A bounded, budget-capped loop

A production loop enforces a hard ceiling on **steps** and **dollars**, independent of what the prompt says. We simulate an agent that wants to keep going but the runtime cuts it off.

In [ ]:
def run_agent(max_steps, max_cost, step_cost=0.002, wants_steps=50, seed=0):
    g = np.random.default_rng(seed)
    spent, steps = 0.0, 0
    while steps < wants_steps:
        if steps >= max_steps:
            return steps, spent, 'hit step ceiling'
        if spent + step_cost > max_cost:
            return steps, spent, 'hit cost ceiling'
        spent += step_cost * (1 + g.random())  # variable tokens per step
        steps += 1
    return steps, spent, 'finished'

print(run_agent(max_steps=10, max_cost=1.0))   # step ceiling bites first
print(run_agent(max_steps=100, max_cost=0.03)) # cost ceiling bites first

### Validate: runtime ceilings bind no matter what the agent wants

The agent "wants" 50 steps, but the runtime must stop it at whichever ceiling — steps
or cost — it hits first. We confirm a tight step ceiling and a tight cost ceiling each
halt the loop early, independent of the agent's intent.

In [ ]:
steps_s, cost_s, why_s = run_agent(max_steps=10, max_cost=1000.0, wants_steps=50)
steps_c, cost_c, why_c = run_agent(max_steps=1000, max_cost=0.05, wants_steps=50)
print(f'tight step ceiling: stopped at {steps_s} steps ({why_s})')
print(f'tight cost ceiling: stopped at {steps_c} steps, ${cost_c:.3f} ({why_c})')
assert steps_s <= 10 and why_s == 'hit step ceiling', 'the step ceiling must bind'
assert cost_c <= 0.05 and why_c == 'hit cost ceiling', 'the cost ceiling must bind'
assert steps_s < 50 and steps_c < 50, 'neither run reaches the agent\'s desired 50 steps'
print('\n✅ the runtime enforces the ceilings — the prompt cannot spend past your budget')

## 2. Cost and latency scale with trajectory length

Each step carries tokens, and steps are sequential, so both cost and latency grow with the number of steps. Parallelising *independent* tool calls cuts latency without cutting work.

In [ ]:
S = np.arange(1, 21)
tokens_per_step = 1500
price = 3 / 1_000_000  # $/token
cost = S * tokens_per_step * price
serial_latency = S * 0.8           # 0.8s/step sequential
parallel_latency = 0.8 * np.ceil(S / 3)  # 3-way parallel where possible

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.plot(S, cost*100, 'o-', color=YELLOW, lw=2); a1.set_title('cost (cents) vs steps')
a1.set_xlabel('steps'); a1.set_ylabel('cents'); a1.grid(True, alpha=0.3)
a2.plot(S, serial_latency, 'o-', color=ROSE, lw=2, label='serial')
a2.plot(S, parallel_latency, 's-', color=TEAL, lw=2, label='3-way parallel')
a2.set_title('latency (s) vs steps'); a2.set_xlabel('steps'); a2.legend(); a2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 3. Checkpointing enables resume

A checkpointed state graph persists after each step, so a crash resumes from the last good step instead of restarting. We mock it with a dict 'store'.

In [ ]:
def stepper(state, store):
    store['checkpoint'] = dict(state)  # persist after each step
    state['step'] += 1
    state['log'].append(f"step {state['step']}")
    return state

store = {}
state = {'step': 0, 'log': []}
for _ in range(3):
    state = stepper(state, store)
print('crash! resuming from checkpoint:', store['checkpoint']['step'])
resumed = dict(store['checkpoint']); resumed['log'] = list(resumed['log'])
resumed = stepper(resumed, store)
print('resumed log:', resumed['log'])

### Validate: checkpointing lets a crashed run resume

Persisting state after each step means a crash loses at most one step. We simulate a
crash mid-run and confirm resuming from the checkpoint continues from where it stopped
rather than restarting from zero.

In [ ]:
store2 = {}
st = {'step': 0, 'log': []}
for _ in range(3):
    st = stepper(st, store2)
crashed_at = store2['checkpoint']['step']   # stepper checkpoints BEFORE incrementing
print(f'checkpoint holds step {crashed_at}')
resumed = dict(store2['checkpoint']); resumed['log'] = list(resumed['log'])
resumed = stepper(resumed, store2)
print(f'resumed to step {resumed["step"]} (not restarted from 0)')
assert crashed_at == 2, 'checkpoint persisted the last step before the crash'
assert resumed['step'] == 3, 'resume continues from the checkpoint, not from scratch'
print('\n✅ checkpointing turns a crash into a one-step loss, not a full restart')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **prompt-level limits only** | the model can ignore "use ≤5 steps"; enforce ceilings in code (verified) |
| **no cost cap** | a looping agent runs up an unbounded bill |
| **no checkpointing** | a crash restarts a long trajectory from zero |
| **latency ≠ cost** | parallelism cuts wall-clock but not token spend (demo) |
| **no idempotency on resume** | replaying side-effecting steps after a crash double-executes them |

Demo: parallelism cuts latency but the token cost is unchanged.

In [ ]:
# Cost is unavoidable and scales linearly with steps; latency can be cut with parallelism
# but cost cannot. We show the gap: 3-way parallel steps finish ~3x faster but cost the same.
S = np.arange(1, 21)
cost = S * 1500 * (3/1_000_000)           # $/step * steps
serial = S * 0.8
parallel = 0.8 * np.ceil(S / 3)
print(f'20 steps: cost ${cost[-1]:.4f}, serial latency {serial[-1]:.1f}s, 3-way parallel {parallel[-1]:.1f}s')
assert parallel[-1] < serial[-1], 'parallelism cuts latency'
# but total token cost is identical whether serial or parallel
assert np.isclose(cost[-1], 20 * 1500 * 3/1_000_000), 'cost scales with total steps, not wall-clock'
print('\nParallelism buys latency, never cost -> to cut cost you must cut STEPS (better planning).')

## ✏️ Your turn — a hard budget cap

Implement `within_budget(spent, step_cost, max_cost)` → `True` only if running one more step keeps total spend ≤ `max_cost`.

In [ ]:
def within_budget(spent, step_cost, max_cost):
    """TODO(you): return whether spent + step_cost <= max_cost."""
    # TODO
    return ...


In [ ]:
assert within_budget(0.00, 0.002, 0.01) is True
assert within_budget(0.009, 0.002, 0.01) is False
assert within_budget(0.008, 0.002, 0.01) is True
print('✅ runtime ceiling enforced regardless of what the prompt asks.')

<details>
<summary>Solution</summary>

```python
def within_budget(spent, step_cost, max_cost):
    return spent + step_cost <= max_cost
```

The prompt is a suggestion; this check is a guarantee. Enforce step *and* dollar ceilings in the runtime, parallelise independent calls, cache, and route easy steps to a cheaper model.
</details>

## Recap

- Enforce **step and cost ceilings in the runtime**, not the prompt.
- Cost and latency scale with trajectory length; **parallelise** independent calls.
- **Checkpointing** turns a crash into a resume and unlocks human-in-the-loop.
- Roll out prompt/tool/model changes with shadow → canary → A/B, like any production system.

## Key takeaways

- **Guardrails belong in the runtime, not the prompt:** hard `max_steps`/`max_cost`
  ceilings bind regardless of what the agent wants (verified).
- **Cost & latency scale with trajectory length;** parallelism cuts latency but *not*
  cost (demo) — reducing steps is the only cost lever.
- **Checkpoint every step** so a crash resumes with a one-step loss, not a restart
  (verified).
- Production agents are as much about **bounding** the loop as about making it smart.